In [ ]:
from langchain_openai import ChatOpenAI

model = ChatOpenAI(
    model="Qwen/Qwen3-8B-MLX-4bit",
    base_url="http://10.195.19.15:8000/v1",
    api_key="dummy",
    temperature=0,
    max_tokens=2048,
)


In [15]:
response = model.invoke("/no_think Say 'stack is alive' if you can hear me.")
print(response.content)



Stack is alive.


## Step 1: load one CSV and inspect it manually

No agent yet — just looking at the raw data so we know its shape before
deciding what the LLM should see (schema + preview only, never the full table).

In [7]:
import pandas as pd

df = pd.read_csv("bazaar_books/caravan_accounts.csv")

print(df.shape)
df.dtypes

(176, 9)


Realm                   str
Guild_Name              str
Year                  int64
Quarter               int64
Operating_Income    float64
EBITDA              float64
Tax                 float64
Net_Income          float64
GOGS                float64
dtype: object

In [8]:
df.head()

,Realm,Guild_Name,Year,Quarter,Operating_Income,EBITDA,Tax,Net_Income,GOGS
0,Oasis of Whispering Sands,Djinn-Forged Ironworks,717,1,21904.87,26090.31,5855.70,16049.17,75668.54
1,Oasis of Whispering Sands,Djinn-Forged Ironworks,717,2,17681.07,23523.18,3967.18,13713.89,62357.84
2,Oasis of Whispering Sands,Djinn-Forged Ironworks,717,3,20009.92,31315.08,4861.25,15148.67,85116.94
3,Oasis of Whispering Sands,Djinn-Forged Ironworks,717,4,14784.45,22956.75,3998.11,10786.34,41932.53
4,Oasis of Whispering Sands,Djinn-Forged Ironworks,718,1,13575.22,16403.08,3558.74,10016.48,35353.81


## Step 2: build the schema-and-preview system prompt

The LLM never sees the full DataFrame. Instead it sees, once, at the start:
- the schema (column names + dtypes) as a markdown table
- a small preview (first N rows) as markdown-KV (key: value pairs) — chosen
  over a markdown table because research shows markdown-KV gives higher
  comprehension accuracy for small models, at the cost of more tokens
  (acceptable here since it's only a handful of rows).

In [ ]:
def build_schema_table(df: pd.DataFrame) -> str:
    """Render column names + dtypes as a markdown table.

    Args:
        df: the DataFrame to describe.

    Returns:
        A markdown table string, one row per column: "column | dtype".
    """
    lines = ["| column | dtype |", "|---|---|"]
    for column_name, dtype in df.dtypes.items():
        lines.append(f"| {column_name} | {dtype} |")
    return "\n".join(lines)


def build_preview_kv(df: pd.DataFrame, n_rows: int = 5) -> str:
    """Render the first n_rows of a DataFrame as markdown-KV blocks.

    Each row becomes a block of "column: value" lines separated by "---".
    Chosen over a markdown table for better small-model comprehension.

    Args:
        df: the DataFrame to preview.
        n_rows: how many rows from the top to include.

    Returns:
        A markdown-KV formatted string.
    """
    blocks = []
    for _, row in df.head(n_rows).iterrows():
        lines = [f"{column_name}: {value}" for column_name, value in row.items()]
        blocks.append("\n".join(lines))
    return "\n---\n".join(blocks)


print(build_schema_table(df))
print()
print(build_preview_kv(df))

In [16]:
SYSTEM_PROMPT_TEMPLATE = """\
You are a financial analytics assistant. You answer questions about a single
pandas DataFrame called `df`, which is already loaded in your execution
environment — you never need to load or recreate it.

You do not have the full table in front of you. You have only the schema
and a small preview below. To answer any question that needs real numbers,
you must call the `execute_python_code` tool with pandas code that operates
on `df` and returns the result. Never guess numeric values — always compute
them via the tool.

## Schema

{schema_table}

## Preview (first {n_preview_rows} rows)

{preview_kv}

## How to answer

- For small talk ("hello", "thank you") — respond directly, do not call the tool.
- For any question needing numbers from the data — write pandas code against
  `df` and call `execute_python_code`. Do not answer from memory or from the
  preview above; the preview is only a sample, not the full data.
"""


def build_system_prompt(df: pd.DataFrame, n_preview_rows: int = 5) -> str:
    """Build the full system prompt for the analytics agent.

    Args:
        df: the DataFrame the agent will answer questions about.
        n_preview_rows: how many rows to include in the preview section.

    Returns:
        The rendered system prompt string.
    """
    return SYSTEM_PROMPT_TEMPLATE.format(
        schema_table=build_schema_table(df),
        n_preview_rows=n_preview_rows,
        preview_kv=build_preview_kv(df, n_preview_rows),
    )


system_prompt = build_system_prompt(df)
print(system_prompt)

You are a financial analytics assistant. You answer questions about a single
pandas DataFrame called `df`, which is already loaded in your execution
environment — you never need to load or recreate it.

You do not have the full table in front of you. You have only the schema
and a small preview below. To answer any question that needs real numbers,
you must call the `execute_python_code` tool with pandas code that operates
on `df` and returns the result. Never guess numeric values — always compute
them via the tool.

## Schema

| column | dtype |
|---|---|
| Realm | str |
| Guild_Name | str |
| Year | int64 |
| Quarter | int64 |
| Operating_Income | float64 |
| EBITDA | float64 |
| Tax | float64 |
| Net_Income | float64 |
| GOGS | float64 |

## Preview (first 5 rows)

Realm: Oasis of Whispering Sands
Guild_Name: Djinn-Forged Ironworks
Year: 717
Quarter: 1
Operating_Income: 21904.87
EBITDA: 26090.31
Tax: 5855.7
Net_Income: 16049.17
GOGS: 75668.54
---
Realm: Oasis of Whispering Sands
Gui

## Step 3: the `execute_python_code` tool

Same idea as the old OpenAI Code Interpreter workflow this project replaces:
the LLM writes pandas code that ends in `print(...)`, the tool runs that
code against the real `df` with `exec()`, captures whatever was printed to
stdout, and returns it as text. If the code raises an exception, we don't
crash — we return the error message back to the agent so it can see what
went wrong and try again with corrected code.

In [17]:
import contextlib
import io

from langchain_core.tools import tool


@tool
def execute_python_code(code: str) -> str:
    """Run pandas code against the loaded financial DataFrame `df` and return its printed output.

    The DataFrame `df` and the `pd` (pandas) module are already available —
    do not try to import pandas or load/recreate `df` yourself.

    Your code MUST call print(...) on whatever value answers the question.
    Anything not printed is lost — this tool only returns what was printed.

    Args:
        code: a snippet of Python/pandas code, e.g.
            "print(df.groupby('Realm')['Net_Income'].sum().idxmax())"

    Returns:
        Everything the code printed to stdout, as a single string. If the
        code raised an exception instead, returns an "Error: ..." message
        describing what went wrong, so you can fix the code and try again.
    """
    namespace = {"df": df, "pd": pd}
    stdout_buffer = io.StringIO()
    try:
        with contextlib.redirect_stdout(stdout_buffer):
            exec(code, namespace)
    except Exception as e:
        return f"Error: {e}"

    output = stdout_buffer.getvalue()
    if not output:
        return "Code ran without errors but printed nothing. Use print() to show a result."
    return output



In [18]:

# quick manual check — no LLM involved, just calling the tool function directly
print(execute_python_code.invoke({"code": "print(df.groupby('Realm')['Net_Income'].sum().idxmax())"}))

Garden of the Midnight Rose

